# CytoBridge synthetic preprocessing

This notebook creates a small spatial count matrix and runs `CytoBridge.pp.preprocess`.
It requires Python 3.10 or later, AnnData, and `CytoBridge[preprocess]` installed in the
current Jupyter kernel.

The example uses synthetic data. Replace the data-construction cell with a dataset loader
that provides raw counts, a time column, cell-type labels, and spatial coordinates.


In [ ]:
from __future__ import annotations

import platform
from importlib.metadata import version

import numpy as np
import pandas as pd
from anndata import AnnData

import CytoBridge

SEED = 42
rng = np.random.default_rng(SEED)
environment = {
    "cytobridge": version("CytoBridge"),
    "python": platform.python_version(),
    "seed": SEED,
}
environment


## 1. Create the input AnnData object

The input contains non-negative integer counts in `layers['counts']`, stage and annotation
columns in `obs`, and two spatial coordinates in `obsm['spatial']`. The assertions check the
array type, range, and dimensions before preprocessing.


In [ ]:
n_cells, n_genes = 72, 40
stages = np.repeat(np.array(["E0", "E1", "E2"]), n_cells // 3)
counts = rng.poisson(2.0, size=(n_cells, n_genes)).astype(np.float32)
counts[stages == "E1", :5] += 2
counts[stages == "E2", 5:10] += 3
counts[:, 39] = 0  # deterministic low-information marker used in the PCA feature check
spatial = np.column_stack([
    np.linspace(0.0, 1.0, n_cells),
    rng.normal(0.0, 0.08, size=n_cells),
]).astype(np.float32)

obs = pd.DataFrame(
    {
        "stage": stages,
        "Annotation": np.where(np.arange(n_cells) % 2 == 0, "TypeA", "TypeB"),
    },
    index=[f"Cell{i:03d}" for i in range(n_cells)],
)
var = pd.DataFrame(index=[f"Gene{i:03d}" for i in range(n_genes)])
adata = AnnData(X=counts.copy(), obs=obs, var=var)
adata.layers["counts"] = counts.copy()
adata.obsm["spatial"] = spatial

assert np.isfinite(counts).all() and (counts >= 0).all()
assert np.allclose(counts, np.rint(counts), rtol=0.0, atol=0.0)
assert adata.obsm["spatial"].shape == (n_cells, 2)
adata.obs.groupby(["stage", "Annotation"], observed=True).size().rename("cells")


## 2. Run preprocessing

`expression_layer='counts'` selects the raw source. Strict validation checks the selected
layer before `normalize_total(target_sum=10000)` and `log1p`. Highly variable genes define
the PCA fit mask without subsetting the expression matrix, and requested features are added
to that mask when needed.


In [ ]:
processed = CytoBridge.pp.preprocess(
    adata.copy(),
    time_key="stage",
    time_mapping={"E0": 0.0, "E1": 1.0, "E2": 2.0},
    n_top_genes=20,
    n_pcs=8,
    expression_layer="counts",
    raw_count_validation="strict",
    required_latent_features=["Gene000", "Gene039"],
)

{
    "X_shape": processed.X.shape,
    "X_latent_shape": processed.obsm["X_latent"].shape,
    "mapped_times": sorted(processed.obs["time_point_processed"].unique().tolist()),
}


## 3. Check preprocessing metadata and arrays

The processed AnnData records the expression source, validation mode, transformation order,
time mapping, PCA feature count, and PCA center in `uns['preprocess_info']`. The checks below
also verify array shapes, finite values, mapped times, and requested PCA features.


In [ ]:
info = processed.uns["preprocess_info"]
assert info["expression_source"] == "layers['counts']"
assert info["raw_count_validation_effective"] == "strict"
assert info["transformation_sequence"] == ["normalize_total", "log1p"]
assert processed.obsm["X_latent"].shape == (n_cells, 8)
assert processed.var["pca_center"].shape == (n_genes,)
assert np.isfinite(processed.obsm["X_latent"]).all()
assert np.isfinite(processed.var["pca_center"].to_numpy()).all()
assert np.allclose(
    processed.obsm["X_latent"].mean(axis=0),
    0.0,
    atol=1e-5,
)
assert sorted(processed.obs["time_point_processed"].unique().tolist()) == [0.0, 1.0, 2.0]
assert all(bool(processed.var.loc[name, "highly_variable"]) for name in ["Gene000", "Gene039"])

preprocess_summary = {
    "expression_source": info["expression_source"],
    "raw_count_validation": info["raw_count_validation_effective"],
    "transformations": info["transformation_sequence"],
    "n_latent_fit_features": info["n_latent_fit_features"],
    "latent_shape": processed.obsm["X_latent"].shape,
    "latent_all_finite": bool(np.isfinite(processed.obsm["X_latent"]).all()),
    "mapped_times": sorted(processed.obs["time_point_processed"].unique().tolist()),
    "required_features_in_pca": ["Gene000", "Gene039"],
}
preprocess_summary


## 4. Validate reuse of a processed object

Passing the transformed matrix through the default preprocessing path a second time raises a
`ValueError`. Start a new preprocessing run from the raw count layer.


In [ ]:
try:
    CytoBridge.pp.preprocess(
        processed.copy(),
        time_key="stage",
        n_top_genes=20,
        n_pcs=8,
    )
except ValueError as exc:
    message = str(exc)
    assert "double-transform" in message
    print(message.splitlines()[0])
else:
    raise AssertionError("Expected preprocessing to reject transformed X")


## Outputs

`processed` contains the normalized expression matrix, `obsm['X_latent']`, PCA metadata,
mapped numeric times, and the original spatial coordinates. `preprocess_summary` collects the
fields checked in this example.
